In [1]:
%load_ext autoreload
%autoreload 2

In [185]:
from concept_abstraction.training import train_model, train_ppo_model
from concept_abstraction.selection import greedy_selection, random_selection, human_centered_selection
from concept_abstraction.env_utils import *
from concept_abstraction.utils import *
from concept_abstraction.environments import *
from sklearn.metrics import accuracy_score
import sys 
import argparse
import secrets
import numpy as np 
import random 
import time 
import gymnasium as gym


In [3]:
is_jupyter = 'ipykernel' in sys.modules

In [4]:
if is_jupyter: 
    seed        = 42
    environment_string = "tree"
    environment_nodes = 127
    show_baseline = True 
    human_accuracy_by_concept = None 
    target_abstraction = 0.05
    out_folder = "synthetic"
    num_concepts_selected = 4
    cbm_accuracy_by_concept = None 
    human_reliance_by_concept = None 
    reward_error = 0
    transition_error = 0
else:
    parser = argparse.ArgumentParser()
    parser.add_argument('--seed', help='Random Seed', type=int, default=42)
    parser.add_argument('--environment_string', help='Which environment to create', type=str, default="tree")
    parser.add_argument('--environment_nodes', help='Size of the environment; number of nodes', type=int, default=4)
    parser.add_argument('--show-baseline', action='store_true', help='Whether to show the baseline')
    parser.add_argument('--num_concepts_selected', help='Number of concepts selected by greedy or random',type=int, default=0)
    parser.add_argument('--human_accuracy_by_concept', nargs='*', type=float, default=None)
    parser.add_argument('--target_abstraction', help='Value for the target abstraction with human performance', type=float, default=0.05)
    parser.add_argument('--cbm_accuracy_by_concept', help="What is the accuracy of AI per concept?", nargs='*', type=float, default=None)
    parser.add_argument('--human_reliance_by_concept', help="How much does AI rely on human intervention?",  nargs='*', type=float, default=None)
    parser.add_argument('--reward_error', help="How much to perturb the reward by?", type=float, default=0)
    parser.add_argument('--transition_error', help="How much to perturb the transition by?", type=float, default=0)
    parser.add_argument('--out_folder', help='Which folder', type=str, default="exploration")

    args = parser.parse_args()

    seed = args.seed
    environment_string = args.environment_string
    environment_nodes = args.environment_nodes 
    show_baseline = args.show_baseline
    num_concepts_selected = args.num_concepts_selected
    human_accuracy_by_concept = args.human_accuracy_by_concept
    human_reliance_by_concept = args.human_reliance_by_concept
    target_abstraction = args.target_abstraction
    cbm_accuracy_by_concept = args.cbm_accuracy_by_concept
    reward_error = args.reward_error
    transition_error = args.transition_error
    out_folder = args.out_folder

save_name = secrets.token_hex(4)  

In [5]:
results = {}
results['parameters'] = {'seed'      : seed,
        'environment_string'    : environment_string, 
        'environment_nodes': environment_nodes, 
        'show_baseline': show_baseline,
        'num_concepts_selected': num_concepts_selected,
        'human_accuracy_by_concept': human_accuracy_by_concept, 
        'human_reliance_by_concept': human_reliance_by_concept, 
        'target_abstraction': target_abstraction,
        'cbm_accuracy_by_concept': cbm_accuracy_by_concept,
        'reward_error': reward_error, 
        'transition_error': transition_error,
}
print("Parameters {}".format(results['parameters']))

Parameters {'seed': 42, 'environment_string': 'tree', 'environment_nodes': 127, 'show_baseline': True, 'num_concepts_selected': 4, 'human_accuracy_by_concept': None, 'human_reliance_by_concept': None, 'target_abstraction': 0.05, 'cbm_accuracy_by_concept': None, 'reward_error': 0, 'transition_error': 0}


In [6]:
np.random.seed(seed)
random.seed(seed)

In [186]:
def make_env_fn(concept_list,accuracies,binary=False,aggregated=False,llm=False):
    env = gym.make("CartPole-v1")

    if binary:
        env = DiscretizeObservationWrapper(env, bins_per_feature=4)
        env = BinaryObservationSubsetWrapper(env, indices=concept_list)
    elif aggregated:
        env = get_binary_subset_env(golden_model, env, concept_list)
    elif llm:
        env = CustomBinaryFeatureWrapper(env)
        env = BinaryObservationSubsetWrapper(env, indices=concept_list)
    else:
        env = ObservationSubsetWrapper(env, indices=concept_list)

    return env

## Retrieving Concepts

In [75]:
def get_average_reward(model,env):
    total_reward = 0

    # Reset and initialize
    observation, info = env.reset()
    for _ in range(10000):
        # Random action: 0 (left) or 1 (right)
        action = model.predict(observation)[0]

        # Take a step in the environment
        observation, reward, terminated, truncated, info = env.step(action)
        total_reward += reward 
        # End episode if done
        if terminated or truncated:
            break
    return total_reward


In [86]:
env = make_env_fn([0,1,2,3],None)
model = train_ppo_model(env,total_timesteps=20000)
get_average_reward(model,env)

361.0

In [85]:
env = make_env_fn([0,1],None)
model = train_ppo_model(env,total_timesteps=20000)
get_average_reward(model,env)

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.8/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


31.0

In [87]:
env = make_env_fn([2,3],None)
model = train_ppo_model(env,total_timesteps=20000)
get_average_reward(model,env)

248.0

In [97]:
env = make_env_fn([0],None,binary=True)
model = train_ppo_model(env,total_timesteps=20000)
get_average_reward(model,env)

40.0

In [96]:
env = make_env_fn(list(range(16)),None,binary=True)
model = train_ppo_model(env,total_timesteps=20000)
get_average_reward(model,env)

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.8/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


251.0

In [100]:
env = make_env_fn([0,1,2,3],None)
golden_model = train_ppo_model(env,total_timesteps=100000)
get_average_reward(golden_model,env)

500.0

In [178]:
env.reset()

(array([0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1]), {})

In [184]:
env = make_env_fn([0,1,2,3],None,aggregated=True)
model = train_ppo_model(env,total_timesteps=20000)
get_average_reward(model,env)

Generating training data for threshold learning...
State ranges:
  cart_pos: [-0.179, 0.111]
  cart_vel: [-0.432, 0.424]
  pole_angle: [-0.047, 0.042]
  pole_vel: [-0.539, 0.516]

cart_pos thresholds:
  20th percentile: -0.099
  40th percentile: -0.061
  60th percentile: -0.030
  80th percentile: 0.002

cart_vel thresholds:
  20th percentile: -0.165
  40th percentile: -0.038
  60th percentile: 0.018
  80th percentile: 0.147

pole_angle thresholds:
  20th percentile: -0.004
  40th percentile: -0.001
  60th percentile: 0.002
  80th percentile: 0.004

pole_vel thresholds:
  20th percentile: -0.232
  40th percentile: -0.037
  60th percentile: 0.027
  80th percentile: 0.235


10.0

In [183]:
env = make_env_fn([0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15],None,aggregated=True)
model = train_ppo_model(env,total_timesteps=20000)
get_average_reward(model,env)

Generating training data for threshold learning...
State ranges:
  cart_pos: [-0.154, 0.094]
  cart_vel: [-0.395, 0.400]
  pole_angle: [-0.044, 0.047]
  pole_vel: [-0.549, 0.516]

cart_pos thresholds:
  20th percentile: -0.064
  40th percentile: -0.025
  60th percentile: -0.010
  80th percentile: 0.013

cart_vel thresholds:
  20th percentile: -0.167
  40th percentile: -0.027
  60th percentile: 0.015
  80th percentile: 0.055

pole_angle thresholds:
  20th percentile: -0.004
  40th percentile: -0.001
  60th percentile: 0.001
  80th percentile: 0.004

pole_vel thresholds:
  20th percentile: -0.170
  40th percentile: -0.040
  60th percentile: 0.009
  80th percentile: 0.239


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.8/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


102.0

In [187]:
env = make_env_fn([0,1,2,3,4,5,6,7,8,9,10,11,12],None,llm=True)
model = train_ppo_model(env,total_timesteps=20000)
get_average_reward(model,env)

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.8/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


453.0

In [188]:
env = make_env_fn([0,1,2],None,llm=True)
model = train_ppo_model(env,total_timesteps=20000)
get_average_reward(model,env)

74.0

## Retrieving Concept Values

In [70]:
if environment_string == 'tree':
    total_timesteps = 40000
else:
    total_timesteps = 10000

In [72]:
baseline_concepts = get_baseline_concept_sets(environment_string,environment_nodes)
results['baseline'] = {'concepts': baseline_concepts}
env = make_env_fn(baseline_concepts[-1],None)

In [73]:
if show_baseline:
    values_by_concept = []
    rewards = []
    transitions = []

    for concept_list in baseline_concepts:
        env = make_env_fn(concept_list,None,reward_error,transition_error)
        model = train_model(env,total_timesteps=total_timesteps)
        env = make_env_fn(concept_list,None)
        values_by_concept.append(get_values(env, model))
        rewards.append(env.rewards.tolist())
        transitions.append(env.transitions.tolist())

    results['baseline'] = {
        'concepts': baseline_concepts,
        'values': values_by_concept,
        'rewards': rewards, 
        'transitions': transitions 
    }

## Concept Selection

In [77]:
selected_concepts = []
random_times = [] 
values_by_random_concept = []
rewards = []
transitions = []
start = time.time() 
for k in range(1,num_concepts_selected+1):
    if k > len(env.concepts):
        break 
    random_concepts = random_selection(env,k)

    env = make_env_fn(random_concepts,None,reward_error=reward_error,transition_error=transition_error)
    model = train_model(env,total_timesteps=total_timesteps)
    selected_concepts.append(random_concepts)
    env = make_env_fn(random_concepts,None)
    values_by_random_concept.append(get_values(env,model))
    random_times.append(time.time()-start)
    rewards.append(env.rewards.tolist())
    transitions.append(env.transitions.tolist())

results['random_selection'] = {
    'concepts': [i.tolist() for i in selected_concepts], 
    'values': values_by_random_concept,
    'time': random_times, 
    'rewards': rewards, 
    'transitions': transitions 
}

In [78]:
selected_concepts = []
values_by_greedy_concept = []
greedy_times = []
rewards = []
transitions = []
start = time.time() 
for k in range(1,num_concepts_selected+1):
    if k > len(env.concepts):
        break 
    greedy_concepts = greedy_selection(env,k)
    env = make_env_fn(greedy_concepts,None,reward_error=reward_error,transition_error=transition_error)
    model = train_model(env,total_timesteps=total_timesteps)
    selected_concepts.append(greedy_concepts)
    env = make_env_fn(greedy_concepts,None)
    values_by_greedy_concept.append(get_values(env,model))
    greedy_times.append(time.time()-start)
    rewards.append(env.rewards.tolist())
    transitions.append(env.transitions.tolist())

results['greedy_selection'] = {
    'concepts': selected_concepts, 
    'values': values_by_greedy_concept,
    'time': greedy_times,
    'rewards': rewards, 
    'transitions': transitions 
}

## Performance under Uncertainty

In [79]:
if human_accuracy_by_concept is not None or cbm_accuracy_by_concept is not None:
    if human_accuracy_by_concept is None:
        modified_acc_rate = cbm_accuracy_by_concept
    elif cbm_accuracy_by_concept is None:
        modified_acc_rate = human_accuracy_by_concept
    else:
        modified_acc_rate = [reliance_percent*human_acc + (1-reliance_percent)*machine_acc 
                for human_acc,machine_acc,reliance_percent in zip(human_accuracy_by_concept,
                                                                cbm_accuracy_by_concept,
                                                                human_reliance_by_concept)]

    selected_concepts = human_centered_selection(env,modified_acc_rate,target_abstraction)
    selected_concepts = [idx for idx,i in enumerate(selected_concepts) if i>=0.5]

    env = make_env_fn(selected_concepts,modified_acc_rate)
    model = train_model(env,total_timesteps=total_timesteps)
    env = make_env_fn(selected_concepts,None)
    human_perf = get_values(env,model)

    concepts = []
    values_error = []

    for idx,concept_list in enumerate(baseline_concepts):
        env = make_env_fn(concept_list,modified_acc_rate)
        model = train_model(env,total_timesteps=total_timesteps)
        concepts.append(concept_list)
        env = make_env_fn(concept_list,None)
        values_error.append(get_values(env,model))

    results['uncertainty'] = {
        'values': values_error,
        'concepts': concepts, 
        'selected_concepts': selected_concepts,
        'combined_accuracies': modified_acc_rate,
        'combined_value': human_perf,
    }

## Save Data

In [80]:
save_path = get_save_path(out_folder,save_name)

In [81]:
delete_duplicate_results(out_folder,"",results)

In [83]:
json.dump(results,open('../../results/'+save_path,'w'))